# سبق 04 - ٹول استعمال کرنے کا ڈیزائن پیٹرن

اس سبق میں آپ مائیکروسافٹ ایجنٹ فریم ورک (پائتھن) استعمال کرتے ہوئے AI ایجنٹس کے لیے **ٹول استعمال کرنے** کا ڈیزائن پیٹرن سیکھیں گے۔ ہم شامل ہیں:

- `@tool` ڈیکوریٹر اور ٹائپ کیے گئے پیرامیٹرز کے ساتھ فنکشن ٹولز کی تعریف کرنا
- ماڈل کو معلوم ہو کہ ہر ٹول کیا کرتا ہے اس کے لیے ٹول سکیمہ فراہم کرنا
- `approval_mode` کے ذریعے ٹول کے نفاذ کو کنٹرول کرنا
- Pydantic ماڈلز اور `response_format` کے ذریعے **مُرتب شدہ آؤٹ پٹ** واپس کرنا

منظر نامہ ایک **سفر بکنگ ایجنٹ** ہے جو مقامات تلاش کر سکتا ہے، دستیابی چیک کر سکتا ہے، اور پرواز کی معلومات حاصل کر سکتا ہے۔


## سیٹ اپ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ڈیکوریٹر کے ساتھ ٹولز کی تعریف کرنا

`@tool` ڈیکوریٹر ایک سادہ پائتھون فنکشن کو ایک ایسے ٹول میں تبدیل کر دیتا ہے جسے ایجنٹ کال کر سکتا ہے۔
اہم نکات:

- **ڈاک سٹرنگ** وہ ٹول کی وضاحت بن جاتی ہے جو ماڈل دیکھتا ہے۔
- **ٹائپ اینوٹیشنز** (جس میں وضاحتوں کے ساتھ `Annotated` شامل ہے) ٹول اسکیمہ کی تعریف کرتی ہیں۔
- `approval_mode` کنٹرول کرتا ہے کہ آیا صارف کو ہر کال سے پہلے اس کی منظوری دینی ہوگی یا نہیں۔


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ایک ایجنٹ بنانا جو متعدد ٹولز استعمال کرے

تمام تینوں ٹولز کو کلائنٹ کو پاس کریں تاکہ ماڈل صارف کے سوال کا جواب دینے کے لیے ضرورت کے مطابق کسی بھی ٹول کو استعمال کر سکے۔


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ٹولز کے ساتھ منظم آؤٹ پٹ

`response_format` کو Pydantic ماڈل پر سیٹ کرنے سے، ایجنٹ پر مجبور کیا جاتا ہے کہ وہ آزاد فارم متن کی بجائے ایک اچھی قسم کا JSON آبجیکٹ فراہم کرے۔ یہ اس وقت مفید ہوتا ہے جب نیچے کے کوڈ کو نتیجہ پروگراماتی طور پر استعمال کرنا ہو۔


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ٹول کی منظوری کے نمونے

`@tool` پر `approval_mode` پیرامیٹر کنٹرول کرتا ہے کہ کیا ٹول کالز کو چلانے سے پہلے انسانی منظوری کی ضرورت ہے:

| موڈ | رویہ |
|---|---|
| `"never_require"` | ٹول خود بخود چلتا ہے — صارف کی تصدیق کی ضرورت نہیں ہے۔ |
| `"always_require"` | ہر کال کو چلانے سے پہلے صارف کی منظوری درکار ہے۔ |

ایسے ٹولز کے لیے `"always_require"` کا استعمال کریں جن کے ضمنی اثرات ہوں (جیسے فلائٹ بک کرنا، کریڈٹ کارڈ چارج کرنا) تاکہ انسان عمل میں شامل رہے۔


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

1. **ٹولز کو متعین کریں** `@tool` ڈیچر کے ساتھ ٹائپڈ پیرامیٹرز اور ڈاک سٹرنگز استعمال کرتے ہوئے جو کہ ٹول کے اسکیما کے طور پر کام کرتے ہیں۔
2. **متعدد ٹولز کو مرتب کریں** تاکہ ایجنٹ انہیں ترتیب سے کال کر کے پیچیدہ سوالات کے جوابات دے سکے۔
3. **مربوط نتائج واپس کریں** پائڈینٹک ماڈل کو `response_format` کے طور پر پاس کر کے۔
4. **ٹول کی منظوری کنٹرول کریں** `approval_mode` کے ساتھ تاکہ حساس کاموں کے لئے انسانی مداخلت برقرار رکھی جا سکے۔

یہ پیٹرنز قابل اعتماد، پروڈکشن کے قابل ایجنٹس بنانے کے لیے بنیاد فراہم کرتے ہیں جو خارجی نظاموں کے ساتھ محفوظ طور پر بات چیت کر سکتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**دیدبان**:  
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم یہ بات مدنظر رکھیں کہ خودکار ترجموں میں غلطیاں یا عدم درستیاں ہوسکتی ہیں۔ اصلی دستاویز اپنی مادری زبان میں مستند ماخذ سمجھی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ ہم اس ترجمہ کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کے ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
